In [1]:
import configparser
import os

import pandas as pd
import pyensembl
from tqdm.notebook import tqdm

In [ ]:
os.chdir('../')

In [2]:
ensembl = pyensembl.EnsemblRelease(75)

In [3]:
dataset = 'all_noHypoHyper'

In [4]:
data = configparser.ConfigParser(interpolation=configparser.ExtendedInterpolation())
data.read('../data.ini')
data = data['DEFAULT']

## Load

In [5]:
dataMutations = pd.read_csv('../' + data[f'{dataset}___mutations'], sep='\t', comment='#', engine='python')
dataMutations_RAS = pd.read_csv('../' + data[f'{dataset}_RAS___mutations'], sep='\t', comment='#', engine='python')
dataMutations_nonRAS = pd.read_csv('../' + data[f'{dataset}_nonRAS___mutations'], sep='\t', comment='#', engine='python')

In [6]:
dataMutations

,Tumor_Sample_Barcode,Hugo_Symbol,original_Hugo_Symbol,Chromosome,Start_Position,End_Position,Strand,Variant_Classification,Variant_Type,Reference_Allele,Tumor_Seq_Allele1,Tumor_Seq_Allele2,HGVSc,HGVSp_Short,Protein_position,Transcript_ID,Gene_Length,Mutation_Length,Protein_Length
0,coadread_dfci_2016_1231,A1BG,A1BG,19,58862793,58862793,+,Missense_Mutation,SNP,C,C,T,ENST00000263100.3:c.874G>A,p.G292R,292.0,ENST00000263100,8322,1,495
1,coadread_dfci_2016_1231,A1BG,A1BG,19,58862794,58862794,+,Silent,SNP,G,G,A,ENST00000263100.3:c.873C>T,p.S291=,291.0,ENST00000263100,8322,1,495
2,coadread_dfci_2016_230737,A1BG,A1BG,19,58858744,58858744,+,Silent,SNP,G,G,A,ENST00000263100.3:c.1455C>T,p.S485=,485.0,ENST00000263100,8322,1,495
3,coadread_dfci_2016_2429,A1BG,A1BG,19,58862934,58862934,+,Missense_Mutation,SNP,G,G,A,ENST00000263100.3:c.733C>T,p.R245C,245.0,ENST00000263100,8322,1,495
4,coadread_dfci_2016_2624,A1BG,A1BG,19,58858848,58858848,+,Missense_Mutation,SNP,C,C,T,ENST00000263100.3:c.1351G>A,p.A451T,451.0,ENST00000263100,8322,1,495
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
388628,TCGA-T9-A92H-01,GPR112,GPR112,X,135428691,135428691,+,Silent,SNP,T,T,G,ENST00000370652.1:c.2826T>G,p.T942=,942.0,ENST00000370652,136094,1,3080
388629,TCGA-T9-A92H-01,DCAF8L2,DCAF8L2,X,27766401,27766401,+,Silent,SNP,T,T,C,ENST00000451261.2:c.1389T>C,p.F463=,463.0,ENST00000451261,158410,1,631
388630,TCGA-T9-A92H-01,MAGEB1,MAGEB1,X,30269452,30269452,+,Missense_Mutation,SNP,C,C,G,ENST00000378981.3:c.842C>G,p.T281S,281.0,ENST00000378981,8309,1,347
388631,TCGA-T9-A92H-01,DMD,DMD,X,32381009,32381009,+,Missense_Mutation,SNP,A,A,C,ENST00000357033.4:c.5221T>G,p.L1741V,1741.0,ENST00000357033,2241765,1,3685


In [7]:
# check that if Tumor_Seq_Allele1 is mutated then also Tumor_Seq_Allele2 is mutated
t1_mut = dataMutations[(dataMutations['Reference_Allele'] != dataMutations['Tumor_Seq_Allele1'])]
t2_nonMut = dataMutations[(dataMutations['Reference_Allele'] == dataMutations['Tumor_Seq_Allele2'])]
t1_mut_t2_nonMut = dataMutations[(dataMutations['Reference_Allele'] != dataMutations['Tumor_Seq_Allele1']) & (dataMutations['Reference_Allele'] == dataMutations['Tumor_Seq_Allele2'])]
t1_mut_t2_difMut = dataMutations[(dataMutations['Reference_Allele'] != dataMutations['Tumor_Seq_Allele1']) & (dataMutations['Reference_Allele'] != dataMutations['Tumor_Seq_Allele2']) & (dataMutations['Tumor_Seq_Allele1'] != dataMutations['Tumor_Seq_Allele2'])]
assert len(t1_mut_t2_nonMut) == 0, "there are mutations where Tumor_Seq_Allele1 is mutated but Tumor_Seq_Allele2 is not"
assert len(t1_mut_t2_difMut) == 0, "there are mutations where Tumor_Seq_Allele1 and Tumor_Seq_Allele2 are mutated but with different mutations"
len(t1_mut), len(t2_nonMut), len(t1_mut_t2_nonMut), len(t1_mut_t2_difMut)

(0, 0, 0, 0)

## Remove non supported data

In [8]:
dataMutations = dataMutations[dataMutations['Mutation_Length'] == 1]
dataMutations_RAS = dataMutations_RAS[dataMutations_RAS['Mutation_Length'] == 1]
dataMutations_nonRAS = dataMutations_nonRAS[dataMutations_nonRAS['Mutation_Length'] == 1]

In [9]:
valid_chr_oncodrive = ['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', 'X', 'Y', 'M']

In [10]:
dataMutations = dataMutations[dataMutations['Chromosome'].isin(valid_chr_oncodrive)]
dataMutations_RAS = dataMutations_RAS[dataMutations_RAS['Chromosome'].isin(valid_chr_oncodrive)]
dataMutations_nonRAS = dataMutations_nonRAS[dataMutations_nonRAS['Chromosome'].isin(valid_chr_oncodrive)]

## Transform data

In [11]:
input_mutations = dataMutations[['Chromosome', 'Start_Position', 'Reference_Allele', 'Tumor_Seq_Allele2', 'Tumor_Sample_Barcode']]
input_mutations = input_mutations.rename(columns={'Chromosome': 'CHROMOSOME', 'Start_Position': 'POSITION', 'Reference_Allele': 'REF', 'Tumor_Seq_Allele2': 'ALT', 'Tumor_Sample_Barcode': 'SAMPLE'})

input_mutations_RAS = dataMutations_RAS[['Chromosome', 'Start_Position', 'Reference_Allele', 'Tumor_Seq_Allele2', 'Tumor_Sample_Barcode']]
input_mutations_RAS = input_mutations_RAS.rename(columns={'Chromosome': 'CHROMOSOME', 'Start_Position': 'POSITION', 'Reference_Allele': 'REF', 'Tumor_Seq_Allele2': 'ALT', 'Tumor_Sample_Barcode': 'SAMPLE'})

input_mutations_nonRAS = dataMutations_nonRAS[['Chromosome', 'Start_Position', 'Reference_Allele', 'Tumor_Seq_Allele2', 'Tumor_Sample_Barcode']]
input_mutations_nonRAS = input_mutations_nonRAS.rename(columns={'Chromosome': 'CHROMOSOME', 'Start_Position': 'POSITION', 'Reference_Allele': 'REF', 'Tumor_Seq_Allele2': 'ALT', 'Tumor_Sample_Barcode': 'SAMPLE'})

In [12]:
input_mutations

,CHROMOSOME,POSITION,REF,ALT,SAMPLE
0,19,58862793,C,T,coadread_dfci_2016_1231
1,19,58862794,G,A,coadread_dfci_2016_1231
2,19,58858744,G,A,coadread_dfci_2016_230737
3,19,58862934,G,A,coadread_dfci_2016_2429
4,19,58858848,C,T,coadread_dfci_2016_2624
...,...,...,...,...,...
388628,X,135428691,T,G,TCGA-T9-A92H-01
388629,X,27766401,T,C,TCGA-T9-A92H-01
388630,X,30269452,C,G,TCGA-T9-A92H-01
388631,X,32381009,A,C,TCGA-T9-A92H-01


In [13]:
def create_annotation_file(genes):
    input_annotations = pd.DataFrame()
    for i, row in tqdm(genes.iterrows(), total=len(genes)):
        gene_id = ensembl.transcript_by_id(row['Transcript_ID']).gene_id
        gene = ensembl.gene_by_id(gene_id)
        annotation_row = pd.DataFrame({'CHROMOSOME': gene.contig, 'START': gene.start, 'END': gene.end, 'ELEMENT': gene.gene_id, 'SYMBOL': gene.gene_name, 'STRAND': gene.strand}, index=[0])
        if row['Chromosome'] != gene.contig:
            print(row['Chromosome'], gene.contig)
        input_annotations = pd.concat([input_annotations, annotation_row], ignore_index=True)
    return input_annotations


In [14]:
input_annotations = create_annotation_file(dataMutations[['Transcript_ID', 'Hugo_Symbol', 'Chromosome']].drop_duplicates())
input_annotations_RAS = create_annotation_file(dataMutations_RAS[['Transcript_ID', 'Hugo_Symbol', 'Chromosome']].drop_duplicates())
input_annotations_nonRAS = create_annotation_file(dataMutations_nonRAS[['Transcript_ID', 'Hugo_Symbol', 'Chromosome']].drop_duplicates())

  0%|          | 0/18982 [00:00<?, ?it/s]

  0%|          | 0/17498 [00:00<?, ?it/s]

  0%|          | 0/18625 [00:00<?, ?it/s]

## Check consistency and save

In [21]:
tmp = dataMutations[['Transcript_ID', 'Hugo_Symbol', 'Chromosome', 'Strand']].merge(input_annotations, left_on='Hugo_Symbol', right_on='SYMBOL')
tmp_RAS = dataMutations_RAS[['Transcript_ID', 'Hugo_Symbol', 'Chromosome', 'Strand']].merge(input_annotations_RAS, left_on='Hugo_Symbol', right_on='SYMBOL')
tmp_nonRAS = dataMutations_nonRAS[['Transcript_ID', 'Hugo_Symbol', 'Chromosome', 'Strand']].merge(input_annotations_nonRAS, left_on='Hugo_Symbol', right_on='SYMBOL')
assert len(tmp[tmp['Chromosome'] != tmp['CHROMOSOME']]) == 0
assert len(tmp_RAS[tmp_RAS['Chromosome'] != tmp_RAS['CHROMOSOME']]) == 0
assert len(tmp_nonRAS[tmp_nonRAS['Chromosome'] != tmp_nonRAS['CHROMOSOME']]) == 0

In [1]:
# TODO: check if there are genes that intersect (same chromosome and considering start and end) in input_annotations

In [17]:
input_mutations.to_csv('../' + data['oncodriveclustl___all_noHypoHyper___in_dir'] + 'mutations.csv', sep='\t', index=False)
input_mutations_RAS.to_csv('../' + data['oncodriveclustl___all_noHypoHyper_RAS___in_dir'] + 'mutations.csv', sep='\t', index=False)
input_mutations_nonRAS.to_csv('../' + data['oncodriveclustl___all_noHypoHyper_nonRAS___in_dir'] + 'mutations.csv', sep='\t', index=False)

input_annotations.to_csv('../' + data['oncodriveclustl___all_noHypoHyper___in_dir'] + 'annotations.csv', sep='\t', index=False)
input_annotations_RAS.to_csv('../' + data['oncodriveclustl___all_noHypoHyper_RAS___in_dir'] + 'annotations.csv', sep='\t', index=False)
input_annotations_nonRAS.to_csv('../' + data['oncodriveclustl___all_noHypoHyper_nonRAS___in_dir'] + 'annotations.csv', sep='\t', index=False)